In [ ]:
#loading required packages
import re
import numpy as np
import tweepy 
from tweepy import OAuthHandler 
from textblob import TextBlob 
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from wordcloud import WordCloud
from better_profanity import profanity
import seaborn
import sklearn
from sklearn.metrics import precision_score,recall_score,accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import TFBertModel, BertTokenizer

main_path = r'C:\Users\Jerry\Desktop\Xacking PhD\dataset\codes\Sentiment_Analysis-master\Sentiment_Analysis-master'

In [ ]:
#reading the dataset
# df = pd.read_csv('ebola_tweets.csv')
df = pd.read_csv(main_path + '/ebola_tweets.csv')

In [ ]:
df.drop_duplicates(subset = ["timestamp", "user", "text"], inplace=True)
print(f"all tweets: {df.shape}")

In [ ]:
df.rename(columns={"text": "tweets"}, inplace=True)
df.head()

In [ ]:
# Converting only the tweets into a list
tweet_list = df.tweets.to_list()
# tweet_list

In [ ]:
# Creating a function to clean the tweets
def clean_tweet(tweet):
    # if type(tweet) == np.float:
    if isinstance(tweet, float):
        return ""
    r = tweet.lower()
    r = profanity.censor(r)
    r = re.sub("'", "", r) # This is to avoid removing contractions in english
    r = re.sub("@[A-Za-z0-9_]+","", r)
    r = re.sub("#[A-Za-z0-9_]+","", r)
    r = re.sub(r'http\S+', '', r)
    r = re.sub('[()!?]', ' ', r)
    r = re.sub('\[.*?\]',' ', r)
    r = re.sub("[^a-z0-9]"," ", r)
    r = re.sub('rt','Mubende',r)
    r = r.split()
    stopwords = ["for", "on", "an", "a", "of", "and", "in", "the", "to", "from","us"]
    r = [w for w in r if not w in stopwords]
    r = " ".join(word for word in r)
    return r

In [ ]:
#cleaning the data
cleaned = [clean_tweet(tw) for tw in tweet_list]

In [ ]:
# Defining the sentiment objects using TextBlob
sentiment_objects = [TextBlob(tweet) for tweet in cleaned]
sentiment_objects[0].polarity, sentiment_objects[0]

In [ ]:
# Creating a list of polarity values and tweet text
sentiment_values = [[tweet.sentiment.polarity, str(tweet)] for tweet in sentiment_objects]
#sentiment_values[0]
# sentiment_values[0:99]

In [ ]:
# Creating a dataframe of each tweet against its polarity
sentiment_df = pd.DataFrame(sentiment_values, columns=["polarity", "tweet"])
sentiment_df.head()

In [ ]:
sentiment_df['label'] = np.where(sentiment_df['polarity'] > 0, 'Positive', np.where(sentiment_df['polarity'] <0, 'Negative', 'Neutral'))

In [ ]:
# # add labeling
# symptom_keywords = [
#     "sore", "fever", "fatigue", "malaise", "muscle pain", "headache", "throat",
#     "vomiting", "diarrhoea", "diarrhea", "abdominal pain", "rash",
#     "kidney", "liver", "ill", "weak", "platelets", "rest", "sick", "fighting", "numb", "flu"
# ]

# # Normalize to lowercase and handle multi-word phrases
# symptom_keywords = [kw.lower() for kw in symptom_keywords]

# # Function to label tweets
# def label_tweet(text):
#     text_lower = text.lower()
#     return "Positive" if any(kw in text_lower for kw in symptom_keywords) else "Negative"

# # Create DataFrame and label tweets labels are done by symptoms and category by sentiment analysis
# sentiment_df['label'] = sentiment_df['tweet'].apply(label_tweet)

In [ ]:
sentiment_df.head()

In [ ]:
# sentiment_df['category'].value_counts()
# sentiment_df['label'].value_counts()

In [ ]:
#category and label encoding
mapping1 = {'Positive':1,'Negative':0,'Neutral':2}
sentiment_df['label'] = sentiment_df['label'].map(mapping1)

In [ ]:
# Store
sentiment_df.to_csv(main_path + "/sentiment_df0.csv", index=False)
sentiment_df.shape

In [ ]:
# Retrieve
preprocessed_df = pd.read_csv(main_path + "/sentiment_df0.csv")
preprocessed_df['tweet'] = preprocessed_df['tweet'].fillna("")
preprocessed_df.shape

In [ ]:
# Feature extraction using bag of words model
tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(preprocessed_df['tweet'])

sequences = tokenizer.texts_to_sequences(preprocessed_df['tweet'])

padded_sequences = pad_sequences(sequences, padding='post', maxlen=None)

# category = preprocessed_df['category']
# labels = preprocessed_df['category']
labels = preprocessed_df['label']

In [ ]:
# padded_sequences.shape, category.shape, labels.shape

In [ ]:
train_data, test_data, train_labels, test_labels = train_test_split(padded_sequences, labels, test_size=0.2, random_state=0)

train_data, val_data, train_labels, val_labels = train_test_split(train_data, train_labels, test_size=0.2, random_state=0)

In [ ]:
train_data.shape, test_data.shape, val_data.shape

In [ ]:
print(f"Size of the training data: {len(train_data)}")
print(f"Size of the training labels data: {len(train_labels)}")
print(f"Size of the validation data: {len(val_data)}")
print(f"Size of the validation lables data: {len(val_labels)}")
print(f"Size of the testing data: {len(test_data)}")
print(f"Size of the testing labels: {len(test_labels)}")

# CNN model

### Model architecture

In [ ]:
model = keras.Sequential([
    keras.layers.Embedding(input_dim=100, output_dim=16, input_length=20),
    keras.layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
    keras.layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
    keras.layers.GlobalMaxPooling1D(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=64, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=256, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=3, activation='softmax')
])

### Model Training

In [ ]:
train_data = pad_sequences(train_data, maxlen=20)
val_data = pad_sequences(val_data, maxlen=20)

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


history = model.fit(train_data, train_labels, epochs=10, verbose=1, batch_size=64, validation_data=(val_data, val_labels))

In [ ]:
model.summary()

### Evaluation plots

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1,len(loss)+1)
plt.plot(epochs,loss,'y',label='Training loss')
plt.plot(epochs,val_loss,'r',label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
plt.plot(epochs,acc,'y',label='Training acc')
plt.plot(epochs,val_acc,'r',label='Validation acc')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

### Model Testing

In [ ]:
test_loss, test_accuracy = model.evaluate(test_data, test_labels)
print("Test Loss:",np.round(test_loss,4))
print("Test Accuracy:", np.round(test_accuracy,4))

In [ ]:
test_data = pad_sequences(test_data, maxlen=20)

predictions = model.predict(test_data)

In [ ]:
predicted_labels = np.argmax(predictions, axis=1)

### Evaluation

In [ ]:
accuracy = accuracy_score(test_labels, predicted_labels)

In [ ]:
precision, recall, f1_score, _ = precision_recall_fscore_support(test_labels, predicted_labels, average="macro")

print("Accuracy:",np.round(accuracy,2))
print("Precision:", np.round(precision,2))
print("Recall:", np.round(recall,2))
print("F1-score:", np.round(f1_score,2))

In [ ]:
# from sklearn.metrics import confusion_matrix
# confusion = confusion_matrix(test_labels, predicted_labels)


# plt.figure(figsize=(8, 6))
# sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", cbar=False)

# plt.title("Confusion Matrix")
# plt.xlabel("Predicted Labels")
# plt.ylabel("True Labels")
# plt.xticks(ticks=[0, 1, 2], labels=["Negative", "Neutral", "Positive"])
# plt.yticks(ticks=[0, 1, 2], labels=["Negative", "Neutral", "Positive"])

# plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
confusion = confusion_matrix(test_labels, predicted_labels)


plt.figure(figsize=(8, 6))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", cbar=False)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.xticks(ticks=[0, 1], labels=["Negative", "Positive"])
plt.yticks(ticks=[0, 1], labels=["Negative", "Positive"])

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

confusion = confusion_matrix(
    test_labels,
    predicted_labels,
    labels=[0, 1]   # Explicitly include only 0 and 1
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False
)

plt.title("Confusion Matrix (Negative vs Positive)")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.xticks(ticks=[0.5, 1.5], labels=["Negative", "Positive"])
plt.yticks(ticks=[0.5, 1.5], labels=["Negative", "Positive"])

plt.tight_layout()
plt.show()
